# PyMIF beginner notebook 00 - installation and mental model

This first notebook is a map of the current PyMIF API. It is meant to reduce confusion caused by older examples that assumed every image was always `TCZYX` and every Zarr was always NGFF v0.4.

The current beginner rule is:

1. Use one of the reader managers to open microscope data, or `ArrayManager` if you already have a NumPy/Dask array.
2. Always make `metadata["axes"]` explicit when creating data yourself.
3. Use `ZarrManager` for modern OME-Zarr reading and writing.
4. Use explicit label metadata with `data_type="label"` when adding segmentations or masks.

## Manager decision table

| Starting point | Beginner class to use | Typical output |
|---|---|---|
| NumPy or Dask array | `ArrayManager` | In-memory PyMIF manager, can write OME-Zarr |
| Existing OME-Zarr v0.4 or v0.5 | `ZarrManager` | Lazy Dask arrays plus groups and labels |
| Legacy NGFF v0.4 only | `ZarrV04Manager` | Compatibility wrapper around `ZarrManager` |
| Luxendo folder | `LuxendoManager` | Lazy TCZYX image pyramid |
| Viventis folder | `ViventisManager` | Lazy TCZYX dataset |
| Opera pyramidal OME-TIFF | `OperaManager` | Lazy TCZYX pyramid |
| Zeiss `.czi` | `ZeissManager` | Lazy TCZYX dataset, scene-aware |
| SCAPE OME-TIFF plus `Metadata/*.xlif` | `ScapeManager` | Lazy TCZYX dataset |

All managers expose the same basic fields:

```python
manager.data        # list of Dask arrays, one per pyramid level
manager.metadata    # normalized PyMIF metadata dictionary
manager.to_zarr()   # write OME-Zarr
manager.build_pyramid()
manager.update_metadata()
manager.subset_dataset()
```

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

## The metadata contract

When you create data yourself, metadata is the most important part. A minimal intensity image usually needs:

```python
metadata = {
    "axes": "tczyx",                  # order of array dimensions
    "size": [(T, C, Z, Y, X)],
    "chunksize": [(1, 1, 8, 512, 512)],
    "scales": [(z_um, y_um, x_um)],   # only spatial axes, in the spatial order from axes
    "units": ("micrometer", "micrometer", "micrometer"),
    "time_increment": 1.0,
    "time_increment_unit": "second",
    "channel_names": ["DAPI", "GFP"],
    "channel_colors": ["0000FF", "00FF00"],
    "dtype": "uint16",
    "data_type": "intensity",
}
```

Key rule: `scales` and `units` only describe spatial axes. If `axes="yx"`, each scale is `(y_spacing, x_spacing)`. If `axes="tzyx"`, each scale is `(z_spacing, y_spacing, x_spacing)`.

In [ ]:
def make_small_raw_manager(seed=0, num_levels=2):
    """Create a tiny TCZYX intensity dataset for examples."""
    rng = np.random.default_rng(seed)
    raw = rng.integers(0, 1200, size=(2, 3, 4, 64, 64), dtype=np.uint16)
    metadata = {
        "name": "small_raw",
        "axes": "tczyx",
        "size": [raw.shape],
        "chunksize": [(1, 1, 2, 32, 32)],
        "scales": [(2.0, 0.30, 0.30)],   # z, y, x spacing because axes contains zyx
        "units": ("micrometer", "micrometer", "micrometer"),
        "time_increment": 60.0,
        "time_increment_unit": "second",
        "channel_names": ["DAPI", "GFP", "RFP"],
        "channel_colors": ["0000FF", "00FF00", "FF0000"],
        "dtype": "uint16",
        "data_type": "intensity",
    }
    manager = mm.ArrayManager(raw, metadata, chunks=metadata["chunksize"][0])
    if num_levels > 1:
        manager.build_pyramid(num_levels=num_levels, downscale_factor=(1, 2, 2))
    return manager

In [ ]:
manager = make_small_raw_manager()
summarize_manager(manager, "Small synthetic raw image")

## Raw, groups, and labels inside a Zarr

A PyMIF OME-Zarr can hold more than one dataset:

```text
my_dataset.zarr/
  0, 1, 2, ...          # raw image pyramid at the root
  denoised/             # optional image group
    0, 1, 2, ...
  labels/
    nuclei/             # optional label group
      0, 1, 2, ...
```

After reading with `ZarrManager`, use the new explicit structure:

```python
z.raw.data                      # root image pyramid
z.raw.metadata                  # root image metadata
z.groups["denoised"].data       # image subgroup
z.groups["denoised"].metadata   # image subgroup metadata
z.labels["nuclei"].data         # label subgroup
z.labels["nuclei"].metadata     # label subgroup metadata
```

The old convenience aliases still exist for the root image:

```python
z.data      # same as z.raw.data
z.metadata  # same as z.raw.metadata
```

## Recommended notebook order

1. `01_read_microscope_datasets_templates.ipynb`
2. `02_array_manager_from_numpy_dask.ipynb`
3. `03_create_ome_zarr_with_metadata.ipynb`
4. `04_read_zarr_inspect_groups_labels.ipynb`
5. `05_add_groups_and_write_image_regions.ipynb`
6. `06_labels_append_inside_existing_zarr.ipynb`
7. `07_create_new_zarr_dataset_with_labels.ipynb`
8. `08_metadata_subset_pyramid_and_channels.ipynb`
9. `09_legacy_options_v04_and_troubleshooting.ipynb`